# Ingest Races CSV File

This notebook demonstrates the ingestion of the races.csv file into the bronze layer.

## What this notebook covers
* Load the races.csv source file
* Add ingestion metadata columns: source_file and ingestion_timestamp
* Write the transformed data to the bronze Delta table
* Validate the loaded results

This uses centralized configuration and helper functions for consistency.

In [0]:
%run ../00-common/01.Formula1_Configuration_Setup

In [0]:
# Check configured catalog
catalog_name

In [0]:
%run ../00-common/02.bronze-helper

In [0]:
# Check landing folder path
landing_folder_path

In [0]:
# Define source file and target table
source_file = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

In [0]:
# Preview source file path
source_file

In [0]:
# Read races CSV into a DataFrame
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
races_schema=StructType([StructField("circuitId",IntegerType()),
                                  StructField("url",StringType()),
                                  StructField("circuitName",StringType()),
                                  StructField("lat",DoubleType()),
                                  StructField("long",DoubleType()),
                                  StructField("locality",StringType()),
                                  StructField("country",StringType())
                                 ])
races_df=spark.read.format("csv").option("header","true").option("mode","FAILFAST").load(source_file)
display(races_df)

In [0]:
# Add ingestion metadata columns
races_final_df=add_ingestion_metadata(races_df)
display(races_final_df)

In [0]:
# Write races data to the bronze Delta table
(
    races_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

In [0]:
# Preview target table name
table_name

In [0]:
%sql
-- Query the bronze races table
select * from formula1.bronze.races

In [0]:
# Display the bronze races table
display(spark.table(table_name))